# 📘 Bölüm 6: RAG ve Agents
## *RAG (Retrieval-Augmented Generation) and Agents*

Kitabın 6. bölümünü Türkçe açıklamalar ile ele alır.

**Ana Konular:**
1. **RAG** (Erişim Destekli Üretim)
2. **Agents** (Ajanlar)
3. **Memory** (Bellek)

> **Temel fikir:** Model hem görevi *nasıl* yapacağını hem de *hangi bilgiyle* yapacağını bilmeli. Bu bölüm, o bağlamın nasıl oluşturulduğunu anlatır.


---
## 1. RAG (Retrieval-Augmented Generation) — Erişim Destekli Üretim

### RAG Nedir?
**RAG**, modelin yanıt üretmeden önce dış bellek kaynaklarından ilgili bilgiyi çektiği bir tekniktir.

- **Retrieval** (Erişim): Dış veri kaynağından ilgili belgeleri getirme
- **Augmentation** (Zenginleştirme): Getirilen bilgiyle bağlamı genişletme  
- **Generation** (Üretim): Zenginleştirilmiş bağlamla yanıt üretme

### RAG Neden Gerekli?
- Modeller eğitim kesim tarihinden sonraki bilgiye sahip değildir
- Tüm bilgiyi **context** (bağlam) içine sığdırmak mümkün değildir
- **Hallucination** (halüsinasyon) riskini azaltır
- **Knowledge-intensive tasks** (bilgi yoğun görevler) için kritiktir

### RAG vs. Uzun Bağlam:
| Yaklaşım | Avantaj | Dezavantaj |
|----------|---------|------------|
| **RAG** | Dinamik, sınırsız veri | Karmaşık altyapı |
| **Long context** (Uzun bağlam) | Basit | Pahalı, dikkat dağınıklığı |

> Anthropic notu: 200.000 token altındaki bilgi tabanları için RAG yerine doğrudan prompt yeterli olabilir.


In [1]:
class SimpleRAGSystem:
    def __init__(self):
        self.documents = {}
        self.chunks = []

    def add_document(self, doc_id, content):
        self.documents[doc_id] = content
        sentences = [s.strip() for s in content.split('.') if s.strip()]
        for i, s in enumerate(sentences):
            self.chunks.append({"doc_id": doc_id, "chunk_id": f"{doc_id}_{i}", "content": s})

    def retrieve(self, query, top_k=3):
        query_words = set(query.lower().split())
        scored = []
        for chunk in self.chunks:
            overlap = len(query_words & set(chunk["content"].lower().split()))
            if overlap > 0:
                scored.append((overlap, chunk))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [c for _, c in scored[:top_k]]

    def run(self, query):
        chunks = self.retrieve(query)
        context = " | ".join([c["content"] for c in chunks])
        return {"query": query, "context": context,
                "prompt": f"Baglamm: {context}\n\nSoru: {query}\n\nYanit:"}

rag = SimpleRAGSystem()
rag.add_document("doc1", "GPT-4 OpenAI tarafindan gelistirilmistir. GPT-4 transformer mimarisi kullanir. GPT-4 2023 yilinda cikti.")
rag.add_document("doc2", "Claude Anthropic tarafindan gelistirildi. Claude guvenlik odakli egitildi.")

result = rag.run("GPT-4 ne zaman cikti")
print("=" * 55)
print("RAG Sistemi Calisma Ornegi")
print("=" * 55)
print(f"Sorgu (Query)  : {result['query']}")
print(f"Baglam (Context): {result['context']}")
print(f"Model Promptu  : {result['prompt']}")


RAG Sistemi Calisma Ornegi
Sorgu (Query)  : GPT-4 ne zaman cikti
Baglam (Context): GPT-4 2023 yilinda cikti | GPT-4 OpenAI tarafindan gelistirilmistir | GPT-4 transformer mimarisi kullanir
Model Promptu  : Baglamm: GPT-4 2023 yilinda cikti | GPT-4 OpenAI tarafindan gelistirilmistir | GPT-4 transformer mimarisi kullanir

Soru: GPT-4 ne zaman cikti

Yanit:


---
## 2. Term-Based Retrieval (Terim Tabanlı Erişim)

**Lexical retrieval** (sözcüksel erişim): Anahtar kelime eşleştirmesine dayalı yöntem.

### TF-IDF (Term Frequency - Inverse Document Frequency):

| Metrik | Açıklama |
|--------|----------|
| **TF** (Terim Sıklığı) | Terimin belgede kaç kez geçtiği |
| **IDF** (Ters Belge Sıklığı) | Nadir terimler daha önemli — `log(N / df)` |

### BM25 (Best Matching 25):
- TF-IDF'in belge uzunluğuna göre normalleştirilmiş gelişmiş hali
- Elasticsearch'ün temelini oluşturur
- Modern sistemlerde hâlâ güçlü bir **baseline** (karşılaştırma noktası)

### Inverted Index (Ters İndeks):
Her terimi, o terimi içeren belgelere eşleştiren sözlük yapısı.

| Terim | Belge Sayısı | (Belge ID, Sıklık) |
|-------|-------------|---------------------|
| python | 4 | (1,5), (10,1) |
| transformer | 3 | (1,5), (38,7) |

### Sparse vs Dense:
- **Sparse** (Seyrek): Çoğunlukla sıfırlardan oluşan vektör (TF-IDF, one-hot)
- **Dense** (Yoğun): Çoğunlukla sıfır olmayan vektör (Embedding'ler)


In [2]:
import math
from collections import defaultdict, Counter

class TFIDFRetriever:
    def __init__(self):
        self.documents = {}
        self.inverted_index = defaultdict(dict)

    def add_document(self, doc_id, text):
        tokens = text.lower().split()
        self.documents[doc_id] = tokens
        for term, count in Counter(tokens).items():
            self.inverted_index[term][doc_id] = count

    def tf(self, term, doc_id):
        tokens = self.documents.get(doc_id, [])
        return tokens.count(term) / len(tokens) if tokens else 0

    def idf(self, term):
        N = len(self.documents)
        df = len(self.inverted_index.get(term, {}))
        return math.log(N / df) if df > 0 else 0

    def score(self, query, doc_id):
        return sum(self.tf(t, doc_id) * self.idf(t) for t in query.lower().split())

    def retrieve(self, query, top_k=3):
        scores = {doc_id: self.score(query, doc_id) for doc_id in self.documents}
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [(doc_id, s) for doc_id, s in ranked[:top_k] if s > 0]


retriever = TFIDFRetriever()
docs = {
    "doc1": "transformer mimarisi derin ogrenme devrim yaratti attention ile",
    "doc2": "python makine ogrenmesi icin yaygin kullanilir scikit",
    "doc3": "transformer elektrik dagitiminda kullanilan bir cihazdır",
    "doc4": "attention mekanizmasi transformer temel bilesendir",
}
for doc_id, text in docs.items():
    retriever.add_document(doc_id, text)

query = "transformer attention"
results = retriever.retrieve(query)
print(f"Sorgu: '{query}'\n")
print("Sirali Belgeler (Ranked Documents):")
print("-" * 55)
for doc_id, score in results:
    print(f"  [{doc_id}] TF-IDF: {score:.4f}")
    print(f"         '{docs[doc_id]}'")
print("\n💡 Terim tabanlı retrieval kelime eslesmesi yapar.")
print("   'transformer' elektrik cihazı belgesini de döndürebilir!")


Sorgu: 'transformer attention'

Sirali Belgeler (Ranked Documents):
-------------------------------------------------------
  [doc4] TF-IDF: 0.1962
         'attention mekanizmasi transformer temel bilesendir'
  [doc1] TF-IDF: 0.1226
         'transformer mimarisi derin ogrenme devrim yaratti attention ile'
  [doc3] TF-IDF: 0.0479
         'transformer elektrik dagitiminda kullanilan bir cihazdır'

💡 Terim tabanlı retrieval kelime eslesmesi yapar.
   'transformer' elektrik cihazı belgesini de döndürebilir!


---
## 3. Embedding-Based Retrieval (Gömü Tabanlı Erişim)

**Semantic retrieval** (anlamsal erişim): Anlam benzerliğine dayalı yöntem.

### Nasıl Çalışır?
```
[Belge] → [Embedding Model] → [Vektör] → [Vector Database]
[Sorgu] → [Embedding Model] → [Vektör] → [Nearest Neighbor Search] → Sonuç
```

### Vector Database (Vektör Veritabanı):
Gömüleri depolayan ve hızlı benzerlik araması yapan özel veritabanı.
**Örnekler:** Pinecone, Weaviate, Chroma, Milvus, pgvector

### ANN (Approximate Nearest Neighbor / Yaklaşık En Yakın Komşu) Algoritmaları:
| Algoritma | Yaklaşım | Özellik |
|-----------|----------|---------|
| **HNSW** (Hierarchical Navigable Small World) | Graf | Yüksek doğruluk + hızlı |
| **LSH** (Locality-Sensitive Hashing) | Hash | Hızlı indeksleme |
| **IVF** (Inverted File Index) | Küme | K-means tabanlı |
| **Product Quantization** | Boyut indirgeme | Düşük bellek kullanımı |
| **Annoy** (Spotify) | Ağaç | Statik veri için |

### Benzerlik Metrikleri (Similarity Metrics):
- **Cosine similarity** (kosinüs benzerliği): Vektör yönleri arası açı, [0,1]
- **Dot product** (nokta çarpımı): Yön + büyüklük birlikte
- **Euclidean distance** (Öklid uzaklığı): Uzaydaki mesafe


In [3]:
import math

def cosine_similarity(v1, v2):
    dot = sum(a*b for a,b in zip(v1,v2))
    n1 = math.sqrt(sum(a**2 for a in v1))
    n2 = math.sqrt(sum(b**2 for b in v2))
    return dot / (n1*n2) if n1 and n2 else 0.0

def euclidean_distance(v1, v2):
    return math.sqrt(sum((a-b)**2 for a,b in zip(v1,v2)))

# Simüle edilmiş gömü vektörleri
# Gerçek senaryoda sentence-transformers gibi kütüphane kullanılır
embeddings = {
    "transformer mimarisi derin ogrenme":  [0.9, 0.8, 0.1, 0.2],
    "attention mekanizmasi ve transformer": [0.85, 0.75, 0.15, 0.25],
    "elektrik transformatoru guc iletimi":  [0.1, 0.2, 0.9, 0.8],
    "python ile makine ogrenmesi":          [0.3, 0.7, 0.2, 0.6],
}
query_vec = [0.88, 0.78, 0.12, 0.22]  # "transformer derin öğrenme" sorgusu

print("Embedding-Based Retrieval (Anlam Benzerligi)")
print("=" * 55)
print(f"Sorgu vektoru: {query_vec}\n")
results = []
for text, vec in embeddings.items():
    cs = cosine_similarity(query_vec, vec)
    ed = euclidean_distance(query_vec, vec)
    results.append((text, cs, ed))
results.sort(key=lambda x: x[1], reverse=True)
for i, (text, cs, ed) in enumerate(results):
    icon = "🥇" if i == 0 else ("🥈" if i == 1 else "  ")
    print(f"{icon} Cos.Sim={cs:.3f} | Oklid={ed:.3f}")
    print(f"   '{text}'")

print("\n💡 'transformer mimarisi' ve 'attention mekanizmasi'")
print("   yakin cunku anlamca iliskili; elektrik belgesi ise uzak.")


Embedding-Based Retrieval (Anlam Benzerligi)
Sorgu vektoru: [0.88, 0.78, 0.12, 0.22]

🥇 Cos.Sim=1.000 | Oklid=0.040
   'transformer mimarisi derin ogrenme'
🥈 Cos.Sim=0.999 | Oklid=0.060
   'attention mekanizmasi ve transformer'
   Cos.Sim=0.812 | Oklid=0.703
   'python ile makine ogrenmesi'
   Cos.Sim=0.359 | Oklid=1.375
   'elektrik transformatoru guc iletimi'

💡 'transformer mimarisi' ve 'attention mekanizmasi'
   yakin cunku anlamca iliskili; elektrik belgesi ise uzak.


---
## 4. Retrieval Evaluation Metrics (Erişim Değerlendirme Metrikleri)

| Metrik | İngilizce | Açıklama |
|--------|-----------|----------|
| **Bağlam Hassasiyeti** | Context Precision | Getirilen belgelerin kaçı gerçekten ilgili? |
| **Bağlam Geri Çağırımı** | Context Recall | İlgili belgelerden kaçı getirildi? |
| **Normalize İnd. Küm. Kazanç** | NDCG | Sıralama kalitesi |
| **Ort. Karşılıklı Sıra** | MRR (Mean Reciprocal Rank) | İlk doğru sonucun ortalama sırası |

### Değerlendirme Araçları:
- **BEIR**: 14 farklı erişim görevi üzerinde değerlendirme çerçevesi
- **MTEB**: Embedding kalitesini çok görevli değerlendirme
- **RAGAS**: RAG sistemleri için özel Python kütüphanesi


In [4]:
def precision_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]
    return len(set(top_k) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]
    return len(set(top_k) & set(relevant)) / len(relevant) if relevant else 0

def reciprocal_rank(retrieved, relevant):
    for i, doc in enumerate(retrieved, 1):
        if doc in relevant:
            return 1.0 / i
    return 0.0

queries = [
    {"query": "transformer mimarisi",
     "retrieved": ["doc_transformer", "doc_attention", "doc_elektrik", "doc_python"],
     "relevant":  ["doc_transformer", "doc_attention"]},
    {"query": "python makine ogrenmesi",
     "retrieved": ["doc_python", "doc_scikit", "doc_transformer", "doc_numpy"],
     "relevant":  ["doc_python", "doc_scikit", "doc_numpy"]},
]

print("Retrieval Degerlendirme Metrikleri")
print("=" * 55)
all_rr = []
for i, q in enumerate(queries, 1):
    ret, rel = q["retrieved"], q["relevant"]
    p3 = precision_at_k(ret, rel, 3)
    r3 = recall_at_k(ret, rel, 3)
    rr = reciprocal_rank(ret, rel)
    all_rr.append(rr)
    rank = int(1/rr) if rr > 0 else "Yok"
    print(f"\nSorgu {i}: '{q['query']}'")
    print(f"  Precision@3:     {p3:.2f}  (3 belgeden {p3*3:.0f} tanesi ilgili)")
    print(f"  Recall@3:        {r3:.2f}  (Ilgili belgelerin %{r3*100:.0f}'i bulundu)")
    print(f"  Reciprocal Rank: {rr:.2f}  (Ilk dogru belge {rank}. sirada)")

mrr = sum(all_rr) / len(all_rr)
print(f"\n🏆 MRR (Mean Reciprocal Rank): {mrr:.3f}")


Retrieval Degerlendirme Metrikleri

Sorgu 1: 'transformer mimarisi'
  Precision@3:     0.67  (3 belgeden 2 tanesi ilgili)
  Recall@3:        1.00  (Ilgili belgelerin %100'i bulundu)
  Reciprocal Rank: 1.00  (Ilk dogru belge 1. sirada)

Sorgu 2: 'python makine ogrenmesi'
  Precision@3:     0.67  (3 belgeden 2 tanesi ilgili)
  Recall@3:        0.67  (Ilgili belgelerin %67'i bulundu)
  Reciprocal Rank: 1.00  (Ilk dogru belge 1. sirada)

🏆 MRR (Mean Reciprocal Rank): 1.000


---
## 5. Hybrid Search ve RAG Optimizasyon Teknikleri

### 5.1 Hybrid Search (Hibrit Arama):
Terim tabanlı ve gömü tabanlı erişimi birleştirir.

**Kombinasyon stratejileri:**
1. **Sequential** (Sıralı): BM25 ile aday havuzu → Semantic ile reranking
2. **Parallel/Ensemble** (Paralel): İkisini aynı anda çalıştır, RRF ile birleştir

**RRF (Reciprocal Rank Fusion / Karşılıklı Sıra Birleştirme):**
$$Score(D) = \sum_{i=1}^{n} \frac{1}{k + r_i(D)}$$

### 5.2 Chunking Strategy (Parçalama Stratejisi):
| Strateji | Açıklama |
|----------|----------|
| **Fixed-size** | Sabit karakter/kelime sayısı |
| **Sentence-based** | Cümle sınırlarına göre |
| **Recursive splitting** | Kademeli bölme (bölüm>paragraf>cümle) |
| **Overlapping chunks** | Örtüşen parçalar (sınır bilgi kaybını önler) |

### 5.3 Reranking (Yeniden Sıralama):
İlk retrieval ucuz ama kaba; cross-encoder reranker hassas ama pahalı.

### 5.4 Query Rewriting (Sorgu Yeniden Yazma):
```
"Emily Doe hakkında ne biliyorsun?"
→ "Emily Doe son satin alma ne zaman?"
```

### 5.5 Contextual Retrieval (Bağlamsal Erişim) — Anthropic Yöntemi:
Her chunk'a orijinal belgenin kısa özetini ekleyerek arama kalitesini artırma.


In [5]:
def rrf(ranked_lists, k=60):
    scores = {}
    for ranked_list in ranked_lists:
        for rank, doc_id in enumerate(ranked_list, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

bm25_results     = ["doc_A", "doc_C", "doc_E", "doc_B", "doc_D"]
semantic_results = ["doc_B", "doc_A", "doc_D", "doc_F", "doc_C"]

fused = rrf([bm25_results, semantic_results])

print("Hybrid Search - RRF (Reciprocal Rank Fusion)")
print("=" * 55)
print("BM25 Siralamasi:    ", bm25_results)
print("Semantic Siralamasi:", semantic_results)
print("\nBirlestirilen Siralama (RRF):")
print(f"{'Sira':>4} | {'Belge':>7} | {'RRF Skor':>10} | {'BM25':>6} | {'Semantic':>8}")
print("-" * 45)
for rank, (doc_id, score) in enumerate(fused, 1):
    b = bm25_results.index(doc_id)+1 if doc_id in bm25_results else "-"
    s = semantic_results.index(doc_id)+1 if doc_id in semantic_results else "-"
    print(f"{rank:>4} | {doc_id:>7} | {score:>10.5f} | {str(b):>6} | {str(s):>8}")

print("\n💡 doc_A: BM25'te 1., semantic'te 2. → En yüksek RRF skoru")
print("   doc_F: Yalnızca semantic'te var → Daha dusuk RRF skoru")


Hybrid Search - RRF (Reciprocal Rank Fusion)
BM25 Siralamasi:     ['doc_A', 'doc_C', 'doc_E', 'doc_B', 'doc_D']
Semantic Siralamasi: ['doc_B', 'doc_A', 'doc_D', 'doc_F', 'doc_C']

Birlestirilen Siralama (RRF):
Sira |   Belge |   RRF Skor |   BM25 | Semantic
---------------------------------------------
   1 |   doc_A |    0.03252 |      1 |        2
   2 |   doc_B |    0.03202 |      4 |        1
   3 |   doc_C |    0.03151 |      2 |        5
   4 |   doc_D |    0.03126 |      5 |        3
   5 |   doc_E |    0.01587 |      3 |        -
   6 |   doc_F |    0.01562 |      - |        4

💡 doc_A: BM25'te 1., semantic'te 2. → En yüksek RRF skoru
   doc_F: Yalnızca semantic'te var → Daha dusuk RRF skoru


---
## 6. RAG Beyond Texts — Metin Ötesi RAG

### 6.1 RAG with Tabular Data (Tablo Verisiyle RAG)

Text-to-SQL iş akışı:
```
[Kullanıcı Sorusu]
       │
       ▼
  [Text-to-SQL]  (LLM ile: Dogal Dil → SQL)
       │
       ▼
  [SQL Execution] (SQL Calistirma)
       │
       ▼
  [Generation] (SQL sonucu + soru → Yanit)
```

### 6.2 Multimodal RAG (Cok Kipli RAG):
- Metin + görüntü + ses kaynaklarından erişim
- **CLIP** gibi çok modlu embedding modelleri kullanılır
- Sorgu → hem metin hem görüntü getirilir

### Context Construction = Feature Engineering:
> RAG'deki baglamm olusturma, klasik ML'deki **feature engineering** ile özdes amac tasir: modele gerekli bilgiyi vermek.


In [6]:
class TextToSQLRAG:
    def __init__(self):
        self.sales = [
            {"id": 1, "product": "Fruity Fedora", "units": 2, "day_ago": 3},
            {"id": 2, "product": "Meow Mix",       "units": 1, "day_ago": 5},
            {"id": 3, "product": "Fruity Fedora", "units": 3, "day_ago": 6},
            {"id": 4, "product": "Purr Shake",    "units": 5, "day_ago": 2},
            {"id": 5, "product": "Fruity Fedora", "units": 1, "day_ago": 8},
            {"id": 6, "product": "Fruity Fedora", "units": 4, "day_ago": 1},
        ]

    def text_to_sql(self, q):
        if "fruity fedora" in q.lower() and "7 gun" in q.lower():
            return "SELECT SUM(units) FROM Sales WHERE product='Fruity Fedora' AND day_ago<=7"
        elif "en cok satan" in q.lower():
            return "SELECT product, SUM(units) FROM Sales GROUP BY product ORDER BY 2 DESC LIMIT 1"
        return "SELECT * FROM Sales LIMIT 5"

    def execute_sql(self, sql):
        from collections import defaultdict
        if "Fruity Fedora" in sql and "day_ago<=7" in sql:
            return sum(r["units"] for r in self.sales if r["product"]=="Fruity Fedora" and r["day_ago"]<=7)
        elif "GROUP BY product" in sql:
            totals = defaultdict(int)
            for r in self.sales: totals[r["product"]] += r["units"]
            return max(totals.items(), key=lambda x: x[1])
        return self.sales[:3]

    def run(self, query):
        print(f"\n{'='*55}")
        print(f"Kullanici: {query}")
        sql = self.text_to_sql(query)
        print(f"1. Text-to-SQL:\n   {sql}")
        result = self.execute_sql(sql)
        print(f"2. SQL Sonucu: {result}")
        if isinstance(result, int):
            print(f"3. Yanit: Son 7 gunde {result} adet Fruity Fedora satildi.")
        elif isinstance(result, tuple):
            print(f"3. Yanit: En cok satan '{result[0]}': {result[1]} adet.")

rag_sql = TextToSQLRAG()
rag_sql.run("Gecen 7 gunde kac tane Fruity Fedora satildi?")
rag_sql.run("En cok satan urun hangisi?")



Kullanici: Gecen 7 gunde kac tane Fruity Fedora satildi?
1. Text-to-SQL:
   SELECT SUM(units) FROM Sales WHERE product='Fruity Fedora' AND day_ago<=7
2. SQL Sonucu: 9
3. Yanit: Son 7 gunde 9 adet Fruity Fedora satildi.

Kullanici: En cok satan urun hangisi?
1. Text-to-SQL:
   SELECT product, SUM(units) FROM Sales GROUP BY product ORDER BY 2 DESC LIMIT 1
2. SQL Sonucu: ('Fruity Fedora', 10)
3. Yanit: En cok satan 'Fruity Fedora': 10 adet.


---
## 7. Agents (Ajanlar)

### Agent Nedir?
> "Çevresini algılayabilen ve bu çevre üzerinde eylemde bulunabilen her şey bir ajandır."
> — Russell & Norvig, *Artificial Intelligence: A Modern Approach*

Bir ajan iki şeyle tanımlanır:
1. **Environment** (Çevre): Ajanın çalıştığı ortam (web, terminal, oyun...)
2. **Actions / Tools** (Eylemler / Araçlar): Ajanın yapabilecekleri

### Araç Kategorileri:
| Kategori | Örnekler | Eylem Türü |
|----------|---------|-----------|
| **Knowledge augmentation** (Bilgi zenginleştirme) | Web arama, SQL, Belge erişimi | Read-only |
| **Capability extension** (Yetenek genişletme) | Hesap makinesi, Kod yorumlayıcı, OCR | Read-only |
| **Write actions** (Yazma eylemleri) | Email gönder, DB güncelle, Banka transferi | Write ⚠️ |

### Function Calling (Fonksiyon Çağırma):
Model sağlayıcılar (OpenAI, Anthropic vb.) araç kullanımı için function calling desteği sunar.

| Mod | Açıklama |
|-----|----------|
| `auto` | Model hangi araçları kullanacağına kendisi karar verir |
| `required` | En az bir araç kullanılmalıdır |
| `none` | Hiçbir araç kullanılmamalıdır |

### Compound Mistakes (Bileşik Hatalar):
Çok adımlı görevlerde hata birikir — güçlü model kritik!


In [7]:
def compound_accuracy(per_step_acc, steps):
    return per_step_acc ** steps * 100

print("Compound Mistakes (Bilesik Hatalar) — Adim Sayisina Gore Dogruluk")
print("=" * 60)
print(f"{'Adim':>6} | {'%99 dogruluk':>14} | {'%95 dogruluk':>14} | {'%90 dogruluk':>14}")
print("-" * 56)
for steps in [1, 3, 5, 10, 20, 50, 100]:
    a99 = compound_accuracy(0.99, steps)
    a95 = compound_accuracy(0.95, steps)
    a90 = compound_accuracy(0.90, steps)
    flag = " <-- dikkat!" if steps == 10 else ""
    print(f"{steps:>6} | {a99:>13.1f}% | {a95:>13.1f}% | {a90:>13.1f}%{flag}")

print("\n⚠️  10 adimda %95 dogruluk → yalnizca %60 genel basari!")
print("   Ajan gorevlerinde guclu model zorunludur.\n")

# Ajan simülasyonu
TOOLS = {
    "get_date":       lambda: "2025-02-25",
    "calculator":     lambda expr: eval(expr),
    "unit_converter": lambda lbs: round(lbs * 0.453592, 3),
    "top_products":   lambda n: ["Fruity Fedora", "Meow Mix", "Purr Shake"][:n],
}

def simple_agent(query):
    print(f"Kullanici: {query}")
    if "kilogram" in query.lower(): plan = [("unit_converter", {"lbs": 40})]
    elif "tarih" in query.lower(): plan = [("get_date", {})]
    elif "satan" in query.lower(): plan = [("top_products", {"n": 3})]
    else: plan = [("calculator", {"expr": "199999/292"})]
    print(f"Plan: {[p[0] for p in plan]}")
    for tool, params in plan:
        result = TOOLS[tool](**params)
        print(f"  {tool}({params}) → {result}")
    print()

simple_agent("40 pound kac kilogram eder?")
simple_agent("En cok satan 3 urunu listele")


Compound Mistakes (Bilesik Hatalar) — Adim Sayisina Gore Dogruluk
  Adim |   %99 dogruluk |   %95 dogruluk |   %90 dogruluk
--------------------------------------------------------
     1 |          99.0% |          95.0% |          90.0%
     3 |          97.0% |          85.7% |          72.9%
     5 |          95.1% |          77.4% |          59.0%
    10 |          90.4% |          59.9% |          34.9% <-- dikkat!
    20 |          81.8% |          35.8% |          12.2%
    50 |          60.5% |           7.7% |           0.5%
   100 |          36.6% |           0.6% |           0.0%

⚠️  10 adimda %95 dogruluk → yalnizca %60 genel basari!
   Ajan gorevlerinde guclu model zorunludur.

Kullanici: 40 pound kac kilogram eder?
Plan: ['unit_converter']
  unit_converter({'lbs': 40}) → 18.144

Kullanici: En cok satan 3 urunu listele
Plan: ['top_products']
  top_products({'n': 3}) → ['Fruity Fedora', 'Meow Mix', 'Purr Shake']



---
## 8. Planning (Planlama) ve ReAct Framework

### Planlama Yöntemleri:
1. **Chain-of-Thought** (Düşünce Zinciri): "Adım adım düşün" → plan + çalıştırma birlikte
2. **Planning-Execution Decoupling** (Ayrıştırma): Önce plan, sonra doğrula, sonra çalıştır
3. **Hierarchical Planning** (Hiyerarşik): Üst düzey plan → alt düzey adımlar

### ReAct Framework *(Yao et al., 2022)*:
**Re**asoning + **Act**ing

```
Thought 1: [Düşünce / Planlama]
Act 1:     [Araç çağrısı]
Observation 1: [Araç çıktısı]
...
Act N: Finish [Son Yanıt]
```

### Reflexion Framework *(Shinn et al., 2023)*:
- **Evaluator**: Sonucu puanla
- **Self-reflection** (Öz-yansıma): Neden başarısız oldun?
- **New trajectory** (Yeni yörünge): Yeni plan üret

### Control Flow (Kontrol Akışı) Türleri:
| Tür | Açıklama |
|-----|----------|
| **Sequential** (Sıralı) | A biter → B başlar |
| **Parallel** (Paralel) | A ve B eş zamanlı |
| **If statement** (Koşullu) | Önceki çıktıya göre B veya C |
| **For loop** (Döngü) | Koşul sağlanana dek tekrar |

### LLM ve Planlama Tartışması:
- Yann LeCun: "Otoregresif LLM'ler planlayamaz"
- Karşı görüş: LLM'ler dünya modeli içeriyor, yeterli araçla planlayabilir


In [8]:
class ReActAgent:
    def __init__(self, tools):
        self.tools = tools
        self.history = []

    def run(self, task, max_steps=4):
        print(f"\n{'='*60}")
        print(f"GOREV: {task}")
        print(f"{'='*60}")
        thoughts = [
            "Gorevi anliyorum. Once bugunun tarihini ogrenmeliyim.",
            "Tarihi ogrendim. Simdi en cok satan urunleri bulabilirim.",
            "Tum bilgileri topladim. Gorevi tamamlayabilirim.",
        ]
        for step in range(max_steps):
            thought = thoughts[step] if step < len(thoughts) else "Dusunuyorum..."
            print(f"\nThought {step+1}: {thought}")
            if step == 0:
                tool, result = "get_date", self.tools["get_date"]()
            elif step == 1:
                tool, result = "top_products", self.tools["top_products"](n=3)
            else:
                print(f"Act {step+1}: Finish")
                h0 = self.history[0]["obs"]
                h1 = self.history[1]["obs"]
                print(f"\nFinal Yanit: {h0} tarihinde en cok satanlar: {h1}")
                break
            print(f"Act {step+1}: {tool}()")
            print(f"Observation {step+1}: {result}")
            self.history.append({"thought": thought, "action": tool, "obs": result})

agent = ReActAgent(tools=TOOLS)
agent.run("Bugunun tarihini ve en cok satan urunleri soyler misin?")



GOREV: Bugunun tarihini ve en cok satan urunleri soyler misin?

Thought 1: Gorevi anliyorum. Once bugunun tarihini ogrenmeliyim.
Act 1: get_date()
Observation 1: 2025-02-25

Thought 2: Tarihi ogrendim. Simdi en cok satan urunleri bulabilirim.
Act 2: top_products()
Observation 2: ['Fruity Fedora', 'Meow Mix', 'Purr Shake']

Thought 3: Tum bilgileri topladim. Gorevi tamamlayabilirim.
Act 3: Finish

Final Yanit: 2025-02-25 tarihinde en cok satanlar: ['Fruity Fedora', 'Meow Mix', 'Purr Shake']


---
## 9. Agent Failure Modes (Ajan Hata Modları) ve Değerlendirme

### Hata Kategorileri:

#### 1. Planning Failures (Planlama Hataları):
| Hata Türü | Örnek |
|-----------|-------|
| **Invalid tool** (Geçersiz araç) | `bing_search` envanterde yok |
| **Invalid parameters** (Geçersiz parametreler) | `unit_converter(40, 'kg')` fazla parametre |
| **Incorrect values** (Hatalı değerler) | `unit_converter(100)` ama 120 gerekiyor |
| **Goal failure** (Hedef başarısızlığı) | Plan hedefi gerçekleştiremiyor |

#### 2. Tool Failures (Araç Hataları):
- Araç yanlış çıktı üretiyor
- Araç erişilemiyor (network hatası)
- Gerekli araç envanterde yok (**missing tool**)

#### 3. Efficiency Failures (Verimlilik Hataları):
- Paralel yapılabilecek şeyleri sıralı yapma
- Gereksiz araç çağrıları

### Agent Benchmark'ları:
- **Berkeley Function Calling Leaderboard**: Fonksiyon çağırma başarısı
- **SWE-bench**: Yazılım geliştirme görevi
- **TravelPlanner Benchmark**: Seyahat planlama görevi


In [9]:
class AgentEvaluator:
    def __init__(self, valid_tools):
        self.valid_tools = valid_tools
        self.logs = []

    def evaluate_plan(self, plan, expected_tools=None):
        errors, warnings = [], []
        for step in plan:
            tool = step.get("tool")
            params = step.get("params", {})
            if tool not in self.valid_tools:
                errors.append(f"INVALID TOOL: '{tool}' envanterde yok")
                continue
            required = set(self.valid_tools[tool].get("params", {}).keys())
            given = set(params.keys())
            if required - given:
                errors.append(f"MISSING PARAMS: '{tool}' icin eksik: {required - given}")
            if given - required:
                warnings.append(f"EXTRA PARAMS: '{tool}' icin fazla: {given - required}")
        if expected_tools:
            used = {s["tool"] for s in plan if s.get("tool") in self.valid_tools}
            missed = set(expected_tools) - used
            if missed:
                warnings.append(f"GOAL RISK: Beklenen araclar kullanilmamis: {missed}")
        return {"valid": len(errors) == 0, "errors": errors, "warnings": warnings}

    def run_suite(self, test_cases):
        valid_count = 0
        for tc in test_cases:
            r = self.evaluate_plan(tc["plan"], tc.get("expected_tools"))
            self.logs.append({"task": tc["task"], **r})
            if r["valid"]: valid_count += 1
        return valid_count, len(test_cases)

    def report(self):
        print("\nAJAN DEGERLENDIRME RAPORU")
        print("=" * 55)
        for log in self.logs:
            status = "✅ GECERLI" if log["valid"] else "❌ GECERSIZ"
            print(f"\nGorev: {log['task']}")
            print(f"Durum: {status}")
            for e in log["errors"]:   print(f"  ❌ {e}")
            for w in log["warnings"]: print(f"  ⚠️  {w}")

tools_def = {
    "get_date":       {"params": {}},
    "top_products":   {"params": {"n": "int"}},
    "unit_converter": {"params": {"lbs": "float"}},
    "calculator":     {"params": {"expr": "str"}},
}

evaluator = AgentEvaluator(valid_tools=tools_def)
test_cases = [
    {"task": "En cok satanlari listele",
     "plan": [{"tool": "get_date", "params": {}},
              {"tool": "top_products", "params": {"n": 3}}],
     "expected_tools": ["top_products"]},
    {"task": "40 pound'u kg'ye cevir",
     "plan": [{"tool": "bing_search", "params": {"query": "lbs to kg"}},
              {"tool": "unit_converter", "params": {"lbs": 40, "unit": "kg"}}],
     "expected_tools": ["unit_converter"]},
    {"task": "199999/292 hesapla",
     "plan": [{"tool": "calculator", "params": {"expr": "199999/292"}}],
     "expected_tools": ["calculator"]},
]
valid, total = evaluator.run_suite(test_cases)
evaluator.report()
print(f"\n📊 OZET: {valid}/{total} plan gecerli ({valid/total*100:.0f}%)")



AJAN DEGERLENDIRME RAPORU

Gorev: En cok satanlari listele
Durum: ✅ GECERLI

Gorev: 40 pound'u kg'ye cevir
Durum: ❌ GECERSIZ
  ❌ INVALID TOOL: 'bing_search' envanterde yok
  ⚠️  EXTRA PARAMS: 'unit_converter' icin fazla: {'unit'}

Gorev: 199999/292 hesapla
Durum: ✅ GECERLI

📊 OZET: 2/3 plan gecerli (67%)


---
## 10. Memory (Bellek)

**Memory**, modelin bilgiyi saklayıp kullanmasını sağlayan mekanizmalar bütünü.

### Üç Bellek Mekanizması:
| Tür | İngilizce | Açıklama | Analoji |
|-----|-----------|----------|---------|
| **Dahili bilgi** | Internal knowledge | Eğitimden gelen | Nefes almak |
| **Kısa süreli** | Short-term memory | Context window | Az önce tanışılan isim |
| **Uzun süreli** | Long-term memory | Harici veri (RAG) | Kitaplar, notlar |

### Bellek Yönetimi Stratejileri:
| Strateji | Avantaj | Dezavantaj |
|----------|---------|------------|
| **FIFO** (First In First Out) | Basit | Önemli bilgi kaybolabilir |
| **Summarization** (Özetleme) | Bilgi korunur | Özetleme maliyeti |
| **Reflection-based** (Yansıma) | Akıllıca seçim | Karmaşık |

### Bellek Yönetiminin Faydaları:
- **Session içi** bilgi taşmasını yönetme
- **Oturumlar arası** kişiselleştirme (tercihleri hatırlama)
- **Tutarlılık**: Önceki yanıtlara referans verebilme


In [10]:
from collections import deque

class MemorySystem:
    def __init__(self, max_tokens=80):
        self.short_term = deque()
        self.long_term = []
        self.max_tokens = max_tokens
        self.total_tokens = 0

    def _tokens(self, text):
        return len(text.split()) * 4 // 3

    def add(self, message, strategy="fifo"):
        msg_tokens = self._tokens(message)
        if self.total_tokens + msg_tokens > self.max_tokens:
            if strategy == "fifo":
                removed = self.short_term.popleft()
                self.total_tokens -= self._tokens(removed)
                self.long_term.append(removed)
                print(f"  [FIFO] Arsivlendi: '{removed[:50]}...'")
            elif strategy == "summarize":
                half = len(self.short_term) // 2
                if half > 0:
                    old = [self.short_term.popleft() for _ in range(half)]
                    for o in old: self.total_tokens -= self._tokens(o)
                    summary = f"[OZET: {len(old)} mesaj]"
                    self.short_term.appendleft(summary)
                    self.total_tokens += self._tokens(summary)
                    print(f"  [OZETLEME] {len(old)} mesaj ozetlendi")
        self.short_term.append(message)
        self.total_tokens += msg_tokens

    def retrieve_long_term(self, keyword):
        return [m for m in self.long_term if keyword.lower() in m.lower()]

    def status(self):
        print(f"  Kisa sureli: {len(self.short_term)} mesaj, ~{self.total_tokens}/{self.max_tokens} token")
        print(f"  Uzun sureli: {len(self.long_term)} arsiv ogesi")

messages = [
    "Kullanici: Merhaba, ben Ahmet. Python calisiyorum.",
    "Asistan: Merhaba Ahmet! Python ile nasil yardimci olabilirim?",
    "Kullanici: Transformer modellerini incelemek istiyorum.",
    "Asistan: HuggingFace kutuphanesi mukemmel bir baslangic.",
    "Kullanici: Hangi modeli onerirsin, BERT mi GPT mi?",
    "Asistan: Siniflandirma icin BERT, uretim icin GPT.",
    "Kullanici: BERT ile baslayayim. Ilk adimlar nelerdir?",
]

print("FIFO Bellek Yonetimi (max 80 token)")
print("=" * 55)
mem = MemorySystem(max_tokens=80)
for msg in messages:
    short = msg[:55] + "..." if len(msg) > 55 else msg
    print(f"\n+ Ekleniyor: '{short}'")
    mem.add(msg, strategy="fifo")

print("\nSon Bellek Durumu:")
mem.status()
print("\nKisa Sureli Bellek Icerigi:")
for m in list(mem.short_term):
    print(f"  - {m[:65]}" + ("..." if len(m) > 65 else ""))
print("\nUzun Sureli Bellekten 'Ahmet' araniyor:")
results = mem.retrieve_long_term("Ahmet")
for r in results:
    print(f"  Bulundu: '{r[:60]}...'")


FIFO Bellek Yonetimi (max 80 token)

+ Ekleniyor: 'Kullanici: Merhaba, ben Ahmet. Python calisiyorum.'

+ Ekleniyor: 'Asistan: Merhaba Ahmet! Python ile nasil yardimci olabi...'

+ Ekleniyor: 'Kullanici: Transformer modellerini incelemek istiyorum.'

+ Ekleniyor: 'Asistan: HuggingFace kutuphanesi mukemmel bir baslangic...'

+ Ekleniyor: 'Kullanici: Hangi modeli onerirsin, BERT mi GPT mi?'

+ Ekleniyor: 'Asistan: Siniflandirma icin BERT, uretim icin GPT.'

+ Ekleniyor: 'Kullanici: BERT ile baslayayim. Ilk adimlar nelerdir?'

Son Bellek Durumu:
  Kisa sureli: 7 mesaj, ~60/80 token
  Uzun sureli: 0 arsiv ogesi

Kisa Sureli Bellek Icerigi:
  - Kullanici: Merhaba, ben Ahmet. Python calisiyorum.
  - Asistan: Merhaba Ahmet! Python ile nasil yardimci olabilirim?
  - Kullanici: Transformer modellerini incelemek istiyorum.
  - Asistan: HuggingFace kutuphanesi mukemmel bir baslangic.
  - Kullanici: Hangi modeli onerirsin, BERT mi GPT mi?
  - Asistan: Siniflandirma icin BERT, uretim icin GPT.
  - 

---
## 11. Bölüm Özeti — Terimler Sözlüğü

| İngilizce Terim | Türkçe | Açıklama |
|----------------|--------|----------|
| **RAG** | Erişim Destekli Üretim | Dış bellek + üretici model |
| **Retrieval** | Erişim | İlgili belgeyi getirme |
| **Chunking** | Parçalama | Belge bölme stratejisi |
| **Indexing** | İndeksleme | Hızlı erişim için veri hazırlama |
| **Term-based retrieval** | Terim tabanlı erişim | Anahtar kelime eşleştirme |
| **Embedding-based retrieval** | Gömü tabanlı erişim | Anlam benzerliğiyle arama |
| **TF-IDF** | Terim Sıklığı × Ters Belge Sıklığı | Önem skoru |
| **BM25** | En İyi Eşleşme 25 | TF-IDF gelişmiş hali |
| **Inverted index** | Ters indeks | Terim → belge eşleme yapısı |
| **Sparse** | Seyrek | Çoğunlukla sıfır vektörler |
| **Dense** | Yoğun | Çoğunlukla sıfır olmayan vektörler |
| **Vector database** | Vektör veritabanı | Gömüleri depolayan özel DB |
| **ANN** | Yaklaşık En Yakın Komşu | Hızlı benzerlik araması |
| **HNSW** | Hiyerarşik Küçük Dünya Grafı | Graf tabanlı ANN |
| **Hybrid search** | Hibrit arama | Terim + gömü birleşimi |
| **RRF** | Karşılıklı Sıra Birleştirme | Sıralamaları birleştirme |
| **Reranking** | Yeniden sıralama | Hassas yeniden sıralama |
| **Query rewriting** | Sorgu yeniden yazma | Belirsiz sorguları açıklama |
| **Contextual retrieval** | Bağlamsal erişim | Chunk'lara bağlam ekleme |
| **Agent** | Ajan | Çevreyi algılayan + eylem yapan sistem |
| **Tool** | Araç | Ajanın kullandığı harici fonksiyon |
| **Function calling** | Fonksiyon çağırma | Model API araç çağrısı |
| **Planning** | Planlama | Adım dizisi belirleme |
| **ReAct** | Akıl Yürüt + Eylem | Düşün > Eylem > Gözlem döngüsü |
| **Reflexion** | Yansıma tabanlı öğrenme | Hatayla öğrenip yeni plan |
| **Compound mistakes** | Bileşik hatalar | Çok adımlı hata birikimi |
| **Text-to-SQL** | Metinden SQL'e | Doğal dil → veritabanı sorgusu |
| **Multimodal RAG** | Çok kipli RAG | Metin + görüntü + ses erişimi |
| **Short-term memory** | Kısa süreli bellek | Context window (sınırlı) |
| **Long-term memory** | Uzun süreli bellek | Harici veri (sınırsız) |
| **FIFO** | İlk Giren İlk Çıkar | Bellek taşma stratejisi |
| **Context precision** | Bağlam hassasiyeti | Getirilen belgelerin isabeti |
| **Context recall** | Bağlam geri çağırımı | Kaçırılmayan ilgili belge oranı |
| **Write actions** | Yazma eylemleri | Veri değiştiren araç eylemleri |
| **Missing tool** | Eksik araç | Gerekli aracın envanterde yokluğu |


https://github.com/chiphuyen/aie-book/raw/main/assets/rag-architecture.png